# Demo - Feature Extraction as Transfer Learning
**Day 3 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/demos/notebooks/demo-feature-extraction.ipynb)

**Goal:** Show that a frozen pre-trained encoder plus a tiny trainable head
reaches good accuracy on a new task without touching the base model weights.

We embed a small labeled set of sentences with a frozen `sentence-transformers`
model (the "backbone" stays fixed), then train only a logistic-regression head
on those features. This is feature extraction: the base does the heavy lifting,
the head learns the last mile. Full fine-tuning, by contrast, would update
every weight in the encoder.

No API key needed. The embedding model downloads once (~80 MB), then caches.

In [1]:
# Scoped install for this demo (safe to re-run).
%pip install -q sentence-transformers scikit-learn

In [2]:
from sklearn.linear_model import LogisticRegression

## 1. A tiny labeled dataset
Ten training sentences (five positive, five negative) and four unseen test
sentences. Labels: 1 = positive, 0 = negative.

In [ ]:
# Tiny labeled sentiment set: 1 = positive, 0 = negative.
TRAIN = [
    ("This product is fantastic, I love it", 1),
    ("Best purchase I have made all year", 1),
    ("Absolutely wonderful and easy to use", 1),
    ("The support team was helpful and kind", 1),
    ("Works exactly as described, very happy", 1),
    ("A complete waste of money", 0),
    ("Terrible quality, broke on day one", 0),
    ("I regret buying this, very disappointed", 0),
    ("The worst experience with any app", 0),
    ("Slow, buggy, and frustrating to use", 0),
]

TEST = [
    "I am thrilled with how well this works",
    "This was a huge letdown and a waste",
    "Delightful and dependable, highly recommend",
    "Awful, do not buy this at all",
]

## 2. Freeze the encoder, embed the training text
The `SentenceTransformer` backbone turns each sentence into a fixed feature
vector. It never updates. Downloads once, then caches.

In [4]:
print("Loading frozen encoder (downloads once, then cached)...")
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [t for t, _ in TRAIN]
labels = [y for _, y in TRAIN]

# The frozen backbone turns text into fixed feature vectors.
train_features = encoder.encode(texts)
print(f"Encoder frozen. Feature vector size: {train_features.shape[1]}")

Loading frozen encoder (downloads once, then cached)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoder frozen. Feature vector size: 384


## 3. Train only a small head
A logistic-regression head learns on top of the frozen features. Only these
few parameters train.

In [5]:
# Only this small head is trained. The encoder never updates.
head = LogisticRegression(max_iter=1000)
head.fit(train_features, labels)
head_params = train_features.shape[1] + 1  # weights + bias
print(f"Trained head parameters: {head_params} (encoder stays frozen)")

Trained head parameters: 385 (encoder stays frozen)


## 4. Predict on unseen sentences
Embed the test sentences with the same frozen encoder and classify with the
trained head.

In [6]:
test_features = encoder.encode(TEST)
preds = head.predict(test_features)
probs = head.predict_proba(test_features)[:, 1]

print("Predictions on unseen sentences:")
for text, pred, prob in zip(TEST, preds, probs):
    label = "positive" if pred == 1 else "negative"
    print(f"  [{label:>8} p={prob:.2f}] {text!r}")

Predictions on unseen sentences:
  [positive p=0.61] 'I am thrilled with how well this works'
  [negative p=0.46] 'This was a huge letdown and a waste'
  [positive p=0.55] 'Delightful and dependable, highly recommend'
  [negative p=0.45] 'Awful, do not buy this at all'


## Expected output
- The feature vector size prints (384 for `all-MiniLM-L6-v2`).
- The head has only a few hundred trainable parameters while the encoder stays
  frozen.
- All four unseen sentences are classified correctly with confident
  probabilities.

**Takeaway:** a frozen base plus a tiny head learns a new task cheaply.